# 0. Setup

In [2]:
import os, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
 
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
# (TensorFlow seed is set later, in the modeling section: tf.random.set_seed(SEED))
 
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

In [3]:
matches = glob.glob("/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/results.csv", recursive=True)
assert len(matches) == 1, f"expected one results.csv, found: {matches}"
DATA_DIR = os.path.dirname(matches[0])
print("DATA_DIR =", DATA_DIR)
print(sorted(os.listdir(DATA_DIR)))

DATA_DIR = /kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020
['circuits.csv', 'constructor_results.csv', 'constructor_standings.csv', 'constructors.csv', 'driver_standings.csv', 'drivers.csv', 'lap_times.csv', 'pit_stops.csv', 'qualifying.csv', 'races.csv', 'results.csv', 'seasons.csv', 'sprint_results.csv', 'status.csv']


In [4]:
TABLES = [
    "circuits", "constructors", "constructor_results", "constructor_standings",
    "drivers", "driver_standings", "races", "results", "qualifying",
    "lap_times", "pit_stops", "sprint_results", "seasons", "status",
]
 
def load_table(name):
    """Read one CSV. Only the literal \\N (and empty cells) become NaN, so legitimate
    strings like 'NA' are not silently turned into missing values."""
    return pd.read_csv(
        os.path.join(DATA_DIR, f"{name}.csv"),
        na_values=[r"\N", ""],
        keep_default_na=False,
    )
 
raw = {name: load_table(name) for name in TABLES}   # bronze: NEVER modify these
df = {name: t.copy() for name, t in raw.items()}    # working copies: all cleaning happens here

In [5]:
summary = pd.DataFrame({
    "rows": {n: len(t) for n, t in raw.items()},
    "cols": {n: t.shape[1] for n, t in raw.items()},
    "missing_cells": {n: int(t.isna().sum().sum()) for n, t in raw.items()},
}).sort_values("rows", ascending=False)
summary

,rows,cols,missing_cells
lap_times,589081,6,0
driver_standings,34863,7,0
results,26759,18,122887
constructor_standings,13391,7,0
constructor_results,12625,5,12608
pit_stops,11371,7,0
qualifying,10494,9,11668
races,1125,18,11349
drivers,861,9,1559
sprint_results,360,16,73
